# Ridge and LASSO Regression

In this assignment we'll look at the affect of using regularization on linear regression models that we train. You will write code to train models that use different regularizers and different penalties to analyze how this affects the model.

---

In [4]:
# Conventionally people rename these common imports for brevity
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Magic command to make the plots appear in-line (it's actually called a "magic command")
%matplotlib inline

In [5]:
# Set seed to create pseudo-randomness
np.random.seed(416)

# Load in the data and preview it
# relative path import for now
home_sales = pd.read_csv('C:/Users/noaht/machine-learning-misc/data/home_data.csv') 
home_sales.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


For this assignment, we will only be using a very small subset of the data to do our analysis. This is not something you would usually do in practice, but is something we do for this assignment to simplify the complexity of this dataset. The data is pretty noisy and to get meaningful results to demonstrate the theoretical behavior, you would need to use a much more complicated set of features that would be a bit more tedious to work with.

In [6]:
# Selects 1% of the data
home_sales = home_sales.sample(frac=0.01, random_state=0) 

print(f'Number of points: {len(home_sales)}')
home_sales.head()

Number of points: 216


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
17384,1453602313,20141029T000000,297000,2,1.50,1430,1650,3.0,0,0,...,7,1430,0,1999,0,98125,47.7222,-122.290,1430,1650
722,2225059214,20140808T000000,1578000,4,3.25,4670,51836,2.0,0,0,...,12,4670,0,1988,0,98005,47.6350,-122.164,4230,41075
2680,2768000270,20140625T000000,562100,2,0.75,1440,3700,1.0,0,0,...,7,1200,240,1914,0,98107,47.6707,-122.364,1440,4300
18754,6819100040,20140624T000000,631500,2,1.00,1130,2640,1.0,0,0,...,8,1130,0,1927,0,98109,47.6438,-122.357,1680,3200
14554,4027700666,20150426T000000,780000,4,2.50,3180,9603,2.0,0,2,...,9,3180,0,2002,0,98155,47.7717,-122.277,2440,15261


## Q1 - Feature Engineering
First, we do a bit of feature engineering by creating features that represent the squares of each feature and the square root of each feature. One benefit of using regularization is you can include more features than necessary and you don't have to be as worried about overfitting since the model is regularized.

In the following cell, complete the code inside the loop to compute the square of each feature the the square root of each feature. *Note*: While we did mention there are some helpful classes in scikit-learn for computing polynomial features, for this assignment we are just computing the square and the square root so we recommend writing the code in pandas yourself.

In [7]:
from math import sqrt

In [9]:
# testing feature extraction
# All of the features of interest
selected_inputs = [
    'bedrooms', 
    'bathrooms',
    'sqft_living', 
    'sqft_lot', 
    'floors', 
    'waterfront', 
    'view', 
    'condition', 
    'grade',
    'sqft_above',
    'sqft_basement',
    'yr_built', 
    'yr_renovated'
]

# Compute the square and sqrt of each feature
# At the end of the for loop: 
#    - sales should have additional columns for the square and 
#      sqrt of each of the selected inputs.
#    - all_features should contain the names (as strings) of all 
#      the features we care about.
all_features = []
for feature_name in selected_inputs:
    
    squared_feature_name = feature_name + '_square'
    sqrt_feature_name = feature_name + '_sqrt'
    
    # compute the square of the column feature_name, add it to sales as a 
    # new column, squared_feature_name
    home_sales[squared_feature_name] = home_sales[feature_name]**2
    
    # compute the sqrt of the column feature_name, add it to sales as a
    # new column, sqrt_feature_name
    home_sales[sqrt_feature_name] = np.sqrt(home_sales[feature_name])
    
    # Add the feature names to all_features
    all_features.extend([feature_name, squared_feature_name, sqrt_feature_name])
    
# Split the data into features and price
price = home_sales['price']
sales = home_sales[all_features]

home_sales.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade_square,grade_sqrt,sqft_above_square,sqft_above_sqrt,sqft_basement_square,sqft_basement_sqrt,yr_built_square,yr_built_sqrt,yr_renovated_square,yr_renovated_sqrt
17384,1453602313,20141029T000000,297000,2,1.50,1430,1650,3.0,0,0,...,49,2.645751,2044900,37.815341,0,0.000000,3996001,44.710178,0,0.0
722,2225059214,20140808T000000,1578000,4,3.25,4670,51836,2.0,0,0,...,144,3.464102,21808900,68.337398,0,0.000000,3952144,44.586994,0,0.0
2680,2768000270,20140625T000000,562100,2,0.75,1440,3700,1.0,0,0,...,49,2.645751,1440000,34.641016,57600,15.491933,3663396,43.749286,0,0.0
18754,6819100040,20140624T000000,631500,2,1.00,1130,2640,1.0,0,0,...,64,2.828427,1276900,33.615473,0,0.000000,3713329,43.897608,0,0.0
14554,4027700666,20150426T000000,780000,4,2.50,3180,9603,2.0,0,2,...,81,3.000000,10112400,56.391489,0,0.000000,4008004,44.743715,0,0.0


it makes sense now why we only took a fraction of the data. With 47 columns, that would've been a lot of compute if we had the whole dataset.

## Split Data
Next, we need to split our data into our train, validation, and test data. To do this, we will use [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) function to split up the dataset. For this assignment we will use 70% of the data to train, 10% for validation, and 20% to test. 

In [10]:
from sklearn.model_selection import train_test_split

In [11]:
# train/validation/test split
sales_train_and_validation, sales_test, price_train_and_validation, price_test = \
    train_test_split(sales, price, test_size=0.2, random_state=6)
sales_train, sales_validation, price_train, price_validation = \
    train_test_split(sales_train_and_validation, price_train_and_validation, test_size=.125, random_state=6) # .10 (validation) of .80 (train + validation)

## Q2 - Standardization

We first need to do a little bit more pre-processing to prepare the data for model training. Models like Ridge and LASSO assume the input features are standardized (mean 0, std. dev. 1) and the target values are centered (mean 0). If we do not do this, we might get some unpredictable results since we violate the assumption of the models!

So in the next cell, you should standardize the data in train, validation, and test using the following instructions:
* Use the [StandardScaler](https://scikit-learn.org/stable/modules/preprocessing.html#standardization-or-mean-removal-and-variance-scaling) preprocessor provided by scikit-learn to do the standardization for you. 
  * Note that you first `fit` it to the data so it can compute the mean/standard deviation and then `transform` to actually change the data. You'll find the examples on this documentation very helpful.
* You should only do this transformation on the features we are using of the data (not any of the other data inputs or the target values). 
* This next note will sound a bit weird, but it's an important step. **You should only do the standardization calculation (e.g., the mean and the standard deviation) on the *training* set and use those statistics to scale the validation and test set**. In other words, the validation and test set should be standardized using the statistics from the training data so that you are using a consistent transformation throughout. This is important to do since you need to apply the same transformation process to every step of the data and you shouldn't use statistics from data outside of your training set in your transformations.

note: LASSO and RIDGE depend on feature scale
only compute mean and std using the training set and reuse that on the validation and test set to avoid data leakage

In [20]:
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler

In [40]:
# preprocess the training, validation, and test data
# use only one scalar fit ONLY on the training data
scaler_train = preprocessing.StandardScaler().fit(sales_train)

In [41]:
# using transform to scale the data
sales_train_standardized = scaler_train.transform(sales_train)
sales_validation_standardized = scaler_train.transform(sales_validation)
sales_test_standardized = scaler_train.transform(sales_test)

recall: all features includes sqrt() and squared features

In [26]:
# train linear regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [48]:
# Train a linear regression model unregularized
model = LinearRegression().fit(sales_train_standardized,price_train)
# sales predictions 
train_estimates = model.predict(sales_train_standardized)
# Train MSE
train_mse = mean_squared_error(price_train, train_estimates)

# train rmse
rmse_test_unregularized = np.sqrt(train_mse)
print(f'Test RMSE: {rmse_test_unregularized:.2f}')

Test RMSE: 142457.32


In [22]:
### edTest(test_ridge) ###

from sklearn.linear_model import Ridge

l2_lambdas = np.logspace(-5, 5, 11, base = 10)

# TODO Implement code to evaluate Ridge Regression with various l2 Penalties
# Note that Ridge uses "alpha" to refer to the variable we have been calling "lambda"

ridge_data = None
ridge_data

Next, let's investigate how the penalty affects the train and validation error by running the following plotting code. 

In [23]:
# Plot the validation RMSE as a blue line with dots
plt.plot(ridge_data['l2_penalty'], ridge_data['rmse_validation'], 
         'b-^', label='Validation')
# Plot the train RMSE as a red line dots
plt.plot(ridge_data['l2_penalty'], ridge_data['rmse_train'], 
         'r-o', label='Train')

# Set y-limits
rmse_max = max(ridge_data['rmse_train'].max(), ridge_data['rmse_validation'].max())
rmse_max *= 1.1  # Give a little buffer
plt.ylim(0, rmse_max)

# Make the x-axis log scale for readability
plt.xscale('log')

# Label the axes and make a legend
plt.xlabel('l2_penalty (log scale)')
plt.ylabel('RMSE')
plt.legend()

Next, we want to actually look at which model we think will perform best. First we define a helper function that will be used to inspect the model parameters.

In [24]:
def print_coefficients(model, features):
    """
    This function takes in a model column and a features column. 
    And prints the coefficient along with its feature name.
    """
    feats = list(zip(features, model.coef_))
    print(*feats, sep = "\n")

## Q5 - Inspecting Coefficients
In the cell below, write code that uses the `ridge_data` `DataFrame` to select which L2 penalty we would choose based on the evaluations we did in the previous section. Do not hard-code the best model parameters, write code to find them from the DataFrame you computed. Compute the following: 

* **Q5.1** -  The best L2 penalty based on the model evaluations. Save this L2 penalty in a variable called `best_l2`.
* **Q5.2** - The best model's error on the **test** dataset. Report the number as an RMSE stored in a variable called `rmse_test_ridge`.
* **Q5.3** The number of coefficients in the best model that are 0. Save this in a variable called `num_zero_coeffs_ridge`. Use the `print_coefficients` function to help you check your result.

Recall that you **may NOT hardcode** values and must instead write code to compute the values; we are testing the data on a slightly different dataset, so hardcoded values from this dataset will be marked as wrong.

Use the next cell answer all three questions. You should also print out the values so you can inspect them.

### Tip

A `pandas` `DataFrame` has a method `idxmin()` function to find the index of the smallest value in a column, and a property `loc` to access a specified index. As an example, suppose we had a `DataFrame` named `df`:

| a | b | c |
|---|---|---|
| 1 | 2 | 3 |
| 2 | 1 | 3 |
| 3 | 2 | 1 |

If we wrote the code 
```python
index = df['b'].idxmin()
row = df.loc[index]
```

It would first find the index of the smallest value in the `b` column and then uses the `.loc` property of the `DataFrame` to access that particular row. It will return a `Series` object (basically a Python dictionary) which means you can use syntax like `row['a']` to access a particular column of that row.

In [25]:
### edTest(test_ridge_analysis) ###

# TODO Print information about best l2 model
best_l2 = None
rmse_test_ridge = None
num_zero_coeffs_ridge = None

# Print your results to help you check their correctness.
print('L2 Penalty',  best_l2)
print('Test RSME', rmse_test_ridge)
print('Num Zero Coeffs', num_zero_coeffs_ridge)

--- 
# LASSO Regression
In this section you will do basically the exact same analysis you did with Ridge Regression, but using LASSO Regression instead. It's okay if your code for this section looks very similar to your code for the last section. 

Remember that for LASSO we choose the parameters that minimize this quality metric instead 

$$\hat{w}_{LASSO} = \min_w MSE(w) + \lambda \left\lVert w \right\rVert_1$$

where $\left\lVert w \right\rVert_1 = \sum_{j=1}^D | w_j |$ is the L1 norm of the parameter vector.

## Q6) Train LASSO Models
We will use the same set of instructions for LASSO as we did for Ridge, except for the following differences. Please refer back to the Ridge Regression instructions and your code to see how these differences fit in!

* Use the [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html#sklearn.linear_model.Lasso) model. Like before, the only parameters you need to pass in are $\lambda$ for the L1 penalty. Like before, sklearn uses the parameter `alpha` instead of `lambda`, but it does the same thing as the `lambda` we discussed in class.
* The range L1 penalties should be $[10, 10^2, ..., 10^7]$. In Python, this is `np.logspace(1, 7, 7, base=10)`.
* The result should be stored in a `DataFrame` named `lasso_data`. All the columns should have the same name and corresponding values except the penalty column should be called `l1_penalty`.
* It is okay if your code prints some `ConvergenceWarning` warnings, these should not impact your results!.

You do not need to worry about your code being redundant with the last section for this part.

In [26]:
### edTest(test_lasso) ###

from sklearn.linear_model import Lasso

l1_lambdas = np.logspace(1, 7, 7, base=10)

# TODO Implement code to evaluate LASSO Regression with various L1 penalties

lasso_data = None
lasso_data

Like before, let's look at how the L1 penalty affects the performance.

In [27]:
# Plot the validation RMSE as a blue line with dots

plt.plot(lasso_data['l1_penalty'], lasso_data['rmse_validation'],
         'b-^', label='Validation')

# Plot the train RMSE as a red line dots
plt.plot(lasso_data['l1_penalty'], lasso_data['rmse_train'],
         'r-o', label='Train')

# Set y-limits
rmse_max = max(lasso_data['rmse_train'].max(), lasso_data['rmse_validation'].max())
rmse_max *= 1.1  # Give a little buffer
plt.ylim(0, rmse_max)

# Make the x-axis log scale for readability
plt.xscale('log')

# Label the axes and make a legend
plt.xlabel('l1_penalty (log scale)')
plt.ylabel('RMSE')
plt.legend()

## Q7 - Inspecting Coefficients
Like before, in the cell below, write code that uses the `lasso_data` `DataFrame` to select which L1 penalty we would choose based on the evaluations we did in the previous section. Do not hard-code the best model paramteres, write code to find the best model from the values you compute from the DataFrame. Compute the following:

* **Q7.1** -  The best L1 penalty based on the model evaluations. Save this L1 penalty in a variable called `best_l1`.
* **Q7.2** - The best model's error on the **test** dataset. Report the number as an RMSE stored in a variable called `rmse_test_lasso`.
* **Q7.3** - The number of coefficients in the best model that are 0. Store this in a variable called `num_zero_coeffs_lasso`. Note that `-0.0` and `0.0` are the same for our purposes. Use the `print_coefficients` function to help you check your result.

In [28]:
### edTest(test_lasso_analysis) ###

# TODO Print information about best L1 model
best_l1 = None
rmse_test_lasso = None
num_zero_coeffs_lasso = None

# Print your results to help you check their correctness.
print('Best L1 Penalty', best_l1)
print('Test RMSE', rmse_test_lasso)
print('Num Zero Coeffs', num_zero_coeffs_lasso)

**Q7.4 -** Let's look at which coefficients ended up having a 0 coefficient. In the cell below, we print the name of all features with coefficient 0. Note, we actually have to check if it is near 0 since numeric computations in Python sometimes yield slight rounding errors (e.g., how 1/3 is .333333333333 and that can't be represented precisely in a computer)


In [1]:
### edTest(test_best_model_lasso) ###

# TODO: Use code like from the above cell to get the best model.
best_model_lasso = None

zero_coef_features = []
nonzero_coef_features = []
for feature, coef in zip(all_features, best_model_lasso.coef_):
  if abs(coef) <= 10 ** -17:
    zero_coef_features.append(feature)
  else:
    nonzero_coef_features.append(feature)

print("Features with coefficient == 0:", zero_coef_features)
print("Features with coefficient != 0:", nonzero_coef_features)